In [87]:
import pandas as pd
import numpy as np

In [88]:
# loading dataset and EDA

In [89]:
import pandas as pd
import numpy as np

In [90]:
chapters = pd.read_csv('chapters.csv')
interactions = pd.read_csv('interactions.csv')

print(chapters.head(5))
print('\n')
print(interactions.head(5))


   chapter_id  chapter_sequence_no  book_id  author_id published_date  \
0     2812946                    1   139726      66847     1990-03-22   
1     4330764                    2   139726      66847     1990-04-09   
2     2664499                    3   139726      66847     1990-04-07   
3     2260666                    4   139726      66847     1990-05-18   
4     6069976                    1   191772      62262     2008-07-30   

                                       tags  
0                            Fantasy|Horror  
1      Fantasy|Young Adult|Literary Fiction  
2                                   Fantasy  
3                  Literary Fiction|Fantasy  
4  Horror|Young Adult|Romance|Graphic Novel  


        user_id  chapter_id  book_id
0  user_2378720     5894067   444295
1  user_2321122     2532511   785684
2  user_2335775     6777764   999595
3  user_7906001     7366896   748410
4  user_9981689     7853186   418083


In [91]:
# Important columns look like chapter_sequence_no, book_id and tags, I need to look more into these columns

In [92]:
# trying to get more about book_id and user id

print(chapters.groupby('book_id').size().describe())
print(interactions.groupby('user_id').size().describe())
print(chapters['book_id'].value_counts())

count    9575.000000
mean        5.221932
std         4.056853
min         1.000000
25%         3.000000
50%         4.000000
75%         6.000000
max        20.000000
dtype: float64
count    149803.000000
mean          6.675434
std           2.572317
min           1.000000
25%           5.000000
50%           6.000000
75%           8.000000
max          20.000000
dtype: float64
book_id
347399    20
712710    20
715357    20
311037    20
536754    20
          ..
105735     1
733698     1
198242     1
486547     1
882186     1
Name: count, Length: 9575, dtype: int64


In [93]:
chapters['chapter_sequence_no'].value_counts()

,count
chapter_sequence_no,
1,9575
2,8631
3,7304
4,5677
5,4216
6,3064
7,2347
8,1744
9,1387


In [94]:
# looking more for chapters and user interaction
user_book_counts = interactions.groupby(['user_id', 'book_id']).size()
print(f"\nChapters read per (user, book) pair:")
print(user_book_counts.describe())
print(f"\n% with exactly 1 chapter: {(user_book_counts == 1).mean() * 100:.2f}%")


Chapters read per (user, book) pair:
count    999520.000000
mean          1.000480
std           0.021909
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           2.000000
dtype: float64

% with exactly 1 chapter: 99.95%


#### 99% of user with exactly one chapter, So we should more focus on Book recommendation instead of chapters recommendation

In [95]:
merged = interactions.merge(
    chapters[['chapter_id', 'book_id', 'chapter_sequence_no', 'author_id', 'tags']],
    on=['chapter_id', 'book_id'], how='left'
)

In [96]:
merged['chapter_sequence_no'].value_counts().sort_index().head(10)
# Chapter sequence numbers read by users

,count
chapter_sequence_no,
1,192094
2,169096
3,145031
4,116760
5,89208
6,63571
7,45782
8,33565
9,26766


In [97]:
# Looking into Genre to find unique tags

In [98]:
# Genre distribution
all_tags = chapters['tags'].dropna().str.split('|').explode()
tag_counts = all_tags.value_counts()
all_tags.nunique()

15

In [99]:
# Genre Distribution
tag_counts

,count
tags,
Crime,8589
Historical Fiction,8547
Dystopian,8448
Adventure,8445
Young Adult,8393
Romance,8389
Science Fiction,8387
Graphic Novel,8376
Paranormal,8359


In [100]:
tag_counts.min()/tag_counts.max()

0.9359646058912563

### Genres are extremely uniform. Individual IDF weights will be near-identical

In [101]:
# Check unique genre combos
book_genre_combos = chapters.groupby('book_id')['tags'].apply(
    lambda x: tuple(sorted(set('|'.join(x.dropna()).split('|')))))
print(f"\nUnique genre combinations: {book_genre_combos.nunique()} across {len(book_genre_combos)} books")


Unique genre combinations: 6907 across 9575 books


In [102]:
# Feature Engineering - TF-IDF Genre Weighting

book_tags = (
    chapters
    .dropna(subset=['tags'])
    .assign(tag_list=lambda df: df['tags'].str.split('|'))
    .groupby('book_id')['tag_list']
    .apply(lambda x: list(set(t for tags in x for t in tags)))
    .reset_index()
)



In [103]:
from sklearn.preprocessing import MultiLabelBinarizer, normalize

mlb = MultiLabelBinarizer()
genre_binary = mlb.fit_transform(book_tags['tag_list'])  # binary TF
genre_df_binary = pd.DataFrame(genre_binary, columns=mlb.classes_, index=book_tags['book_id'])

n_total_books = len(book_tags)
doc_freq = genre_binary.sum(axis=0)  # how many books have each genre
idf_weights = np.log(n_total_books / (doc_freq + 1))  # +1 smoothing

print("IDF weights per genre:")
for genre, idf in sorted(zip(mlb.classes_, idf_weights), key=lambda x: x[1]):
    print(f"  {genre:<20} IDF = {idf:.4f}")

# Apply TF-IDF: element-wise multiply binary vectors by IDF
genre_tfidf = genre_binary * idf_weights
genre_df = pd.DataFrame(genre_tfidf, columns=mlb.classes_, index=book_tags['book_id'])

print(f"\nTF-IDF genre matrix: {genre_df.shape} (books × genres)")


IDF weights per genre:
  Graphic Novel        IDF = 0.8061
  Young Adult          IDF = 0.8082
  Historical Fiction   IDF = 0.8087
  Thriller             IDF = 0.8087
  Horror               IDF = 0.8115
  Adventure            IDF = 0.8158
  Paranormal           IDF = 0.8186
  Dystopian            IDF = 0.8191
  Literary Fiction     IDF = 0.8205
  Romance              IDF = 0.8205
  Crime                IDF = 0.8245
  Humor                IDF = 0.8293
  Science Fiction      IDF = 0.8293
  Fantasy              IDF = 0.8310
  Mystery              IDF = 0.8310

TF-IDF genre matrix: (9575, 15) (books × genres)


In [104]:
book_meta = chapters.groupby('book_id').agg(
    author_id=('author_id', 'first'),
    num_chapters=('chapter_id', 'count'),
    first_published=('published_date', 'min'),
).reset_index()

book_meta.head()

,book_id,author_id,num_chapters,first_published
0,100089,55121,3,1995-03-24
1,100096,61876,1,2015-12-31
2,100193,67983,4,1996-01-26
3,100205,64099,4,2016-02-26
4,100301,77538,3,2015-08-14


In [105]:
## Making user book interaction metrics for model 1

In [106]:

user_book = interactions.groupby(['user_id', 'book_id']).size().reset_index(name='count')
user_book['count'] = 1

# Create index mappings
user_ids = sorted(user_book['user_id'].unique())
book_ids = sorted(user_book['book_id'].unique())
user_to_idx = {u: i for i, u in enumerate(user_ids)}
book_to_idx = {b: i for i, b in enumerate(book_ids)}
idx_to_user = {i: u for u, i in user_to_idx.items()}
idx_to_book = {i: b for b, i in book_to_idx.items()}

n_users = len(user_ids)
n_books = len(book_ids)

rows = user_book['user_id'].map(user_to_idx).values
cols = user_book['book_id'].map(book_to_idx).values
data = user_book['count'].values

interaction_matrix = sparse.csr_matrix((data, (rows, cols)), shape=(n_users, n_books))
print(f"Interaction matrix: {interaction_matrix.shape}")
print(f"Density: {interaction_matrix.nnz / (n_users * n_books):.6f}")
print(f"Total interactions: {interaction_matrix.nnz:,}")


Interaction matrix: (149803, 9575)
Density: 0.000697
Total interactions: 999,520


In [107]:
# pre computing max chapter per user and book par and book list for quick next-chapter lookup

In [108]:

user_book_max_chapter = (
    merged.groupby(['user_id', 'book_id'])['chapter_sequence_no']
    .max()
    .to_dict()
)


In [109]:
book_chapter_map = {}
for book_id, group in chapters.sort_values('chapter_sequence_no').groupby('book_id'):
    book_chapter_map[book_id] = list(
        zip(group['chapter_sequence_no'].values, group['chapter_id'].values)
    )

print(f"Pre-computed chapter lookups: {len(user_book_max_chapter):,} user-book pairs")
print(f"Book chapter maps: {len(book_chapter_map):,} books")


Pre-computed chapter lookups: 999,520 user-book pairs
Book chapter maps: 9,575 books


In [110]:

# defining function for quick lookup
def recommend_chapter_fast(user_id, book_id):
    max_read = user_book_max_chapter.get((user_id, book_id), 0)

    if book_id not in book_chapter_map:
        return None, None

    chapter_list = book_chapter_map[book_id]

    if max_read == 0:
        # New book -> chapter 1
        return chapter_list[0][1], chapter_list[0][0]

    # Find next chapter after max_read
    for seq_no, ch_id in chapter_list:
        if seq_no > max_read:
            return ch_id, seq_no

    return None, None  # finished the book

In [111]:
# train - test split
#Leave-one-out — hold out 1 random book per user (for users with ≥2 books)

In [112]:
from scipy import sparse
from scipy.sparse.linalg import svds

In [113]:
user_books_list = user_book.groupby('user_id')['book_id'].apply(list).to_dict()

train_pairs = []
test_pairs = []
cold_users = []

for user, books in user_books_list.items():
    if len(books) >= 2:
        test_book = np.random.choice(books)
        test_pairs.append((user, test_book))
        train_books = [b for b in books if b != test_book]
        for b in train_books:
            train_pairs.append((user, b))
    else:
        cold_users.append(user)
        for b in books:
            train_pairs.append((user, b))

train_df = pd.DataFrame(train_pairs, columns=['user_id', 'book_id'])
test_df = pd.DataFrame(test_pairs, columns=['user_id', 'book_id'])

print(f"Train interactions: {len(train_df):,}")
print(f"Test interactions:  {len(test_df):,} (1 per evaluable user)")
print(f"Cold users (1 book, train-only): {len(cold_users):,}")

# Build train interaction matrix
train_rows = train_df['user_id'].map(user_to_idx).values
train_cols = train_df['book_id'].map(book_to_idx).values
train_matrix = sparse.csr_matrix(
    (np.ones(len(train_df)), (train_rows, train_cols)),
    shape=(n_users, n_books)
)


Train interactions: 850,972
Test interactions:  148,548 (1 per evaluable user)
Cold users (1 book, train-only): 1,255


## Model 1: Collaborative Filtering (Truncated SVD)

Matrix factorization via truncated SVD on the user-book interaction matrix. Captures latent "users like you also read X" patterns.


In [114]:
K = 50

U, sigma, Vt = svds(train_matrix.astype(float), k=K)
sigma_diag = np.diag(sigma)

# User and item latent factors
user_factors = U @ sigma_diag
item_factors = Vt.T

print(f"Latent factors computed: users={user_factors.shape}, items={item_factors.shape}")


def cf_scores(user_idx, train_mat, n_recs=100):
    """Get CF scores for a user. Returns dict of {book_idx: score}."""
    scores = user_factors[user_idx] @ item_factors.T
    # Zero out already-read books
    read_books = set(train_mat[user_idx].nonzero()[1])
    for b in read_books:
        scores[b] = -np.inf
    top_indices = np.argsort(scores)[::-1][:n_recs]
    return {idx: scores[idx] for idx in top_indices}


Latent factors computed: users=(149803, 50), items=(9575, 50)


In [115]:
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

## Model 2: Content-Based Filtering (TF-IDF Genre Similarity)

User preference profile = normalized mean of TF-IDF genre vectors of their read books. Score unread books by cosine similarity.



In [116]:
# Align TF-IDF genre vectors to book index ordering
genre_vectors = np.zeros((n_books, len(mlb.classes_)))
for book_id, idx in book_to_idx.items():
    if book_id in genre_df.index:
        genre_vectors[idx] = genre_df.loc[book_id].values

genre_vectors_norm = normalize(genre_vectors, norm='l2', axis=1)
print(f"TF-IDF genre vectors: {genre_vectors_norm.shape}")


def content_scores(user_idx, train_mat, n_recs=100):

    read_books = train_mat[user_idx].nonzero()[1]
    if len(read_books) == 0:
        return {}

    # User profile = mean TF-IDF genre vector of read books
    user_profile = genre_vectors_norm[read_books].mean(axis=0).reshape(1, -1)
    user_profile = normalize(user_profile, norm='l2')

    scores = cosine_similarity(user_profile, genre_vectors_norm).flatten()

    for b in read_books:
        scores[b] = -np.inf

    top_indices = np.argsort(scores)[::-1][:n_recs]
    return {idx: scores[idx] for idx in top_indices}


TF-IDF genre vectors: (9575, 15)


## Model 3: Hybrid Recommender

Adaptive weighted blend of CF and Content-Based scores:
- **Cold users (≤2 books)**: α=0.3 (lean content-based — CF can't generalize from 1-2 data points)
- **Medium users (3-5 books)**: α=0.5 (balanced)
- **Warm users (5+ books)**: α=0.7 (lean CF — enough collaborative signal)

Scores from each model are min-max normalized before blending.

In [117]:
def hybrid_recommend(user_idx, train_mat, n_recs=10, alpha_cf=0.7):
    n_read = train_mat[user_idx].nnz

    # Adaptive blending
    if n_read <= 2:
        alpha = 0.3
    elif n_read <= 5:
        alpha = 0.5
    else:
        alpha = alpha_cf

    cf = cf_scores(user_idx, train_mat, n_recs=200)
    cb = content_scores(user_idx, train_mat, n_recs=200)

    def normalize_scores(scores):
        if not scores:
            return scores
        vals = np.array(list(scores.values()))
        valid = vals[vals > -np.inf]
        if len(valid) == 0 or valid.max() == valid.min():
            return {k: 0.0 for k, v in scores.items() if v > -np.inf}
        mn, mx = valid.min(), valid.max()
        return {k: (v - mn) / (mx - mn) for k, v in scores.items() if v > -np.inf}

    cf_norm = normalize_scores(cf)
    cb_norm = normalize_scores(cb)

    all_books = set(cf_norm.keys()) | set(cb_norm.keys())
    combined = {}
    for b in all_books:
        s_cf = cf_norm.get(b, 0.0)
        s_cb = cb_norm.get(b, 0.0)
        combined[b] = alpha * s_cf + (1 - alpha) * s_cb

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:n_recs]
    return [(idx_to_book[b], score) for b, score in ranked]


## Cold-Start Handling

For brand-new users with zero history, popularity-based recommendations optionally filtered by a stated genre preference.


In [118]:
book_popularity = (
    train_df.groupby('book_id').size()
    .reset_index(name='read_count')
    .sort_values('read_count', ascending=False)
)

#Recommend for a brand new user. Optionally filter by genre
def cold_start_recommend(genre_preference=None, n_recs=10):

    if genre_preference and genre_preference in genre_df_binary.columns:
        matching_books = genre_df_binary[genre_df_binary[genre_preference] == 1].index
        recs = (
            book_popularity[book_popularity['book_id'].isin(matching_books)]
            .head(n_recs)['book_id'].tolist()
        )
    else:
        recs = book_popularity.head(n_recs)['book_id'].tolist()
    return recs


print("Top 10 most popular books (cold-start fallback):")
top_popular = book_popularity.head(10).merge(book_meta, on='book_id')
for _, row in top_popular.iterrows():
    bgs = genre_df_binary.loc[row['book_id']] if row['book_id'] in genre_df_binary.index else {}
    top_g = [g for g, v in bgs.items() if v == 1] if isinstance(bgs, pd.Series) else []
    print(f"  Book {row['book_id']:>7}  |  reads: {row['read_count']:>4}  |  "
          f"chapters: {row['num_chapters']}  |  genres: {', '.join(top_g[:4])}")


Top 10 most popular books (cold-start fallback):
  Book  672233  |  reads:  999  |  chapters: 20  |  genres: Adventure, Crime, Dystopian, Fantasy
  Book  760434  |  reads:  818  |  chapters: 16  |  genres: Adventure, Crime, Dystopian, Fantasy
  Book  615276  |  reads:  807  |  chapters: 19  |  genres: Adventure, Crime, Fantasy, Graphic Novel
  Book  756944  |  reads:  772  |  chapters: 16  |  genres: Adventure, Crime, Dystopian, Fantasy
  Book  395485  |  reads:  708  |  chapters: 8  |  genres: Crime, Graphic Novel, Historical Fiction, Horror
  Book  682667  |  reads:  697  |  chapters: 20  |  genres: Adventure, Crime, Dystopian, Fantasy
  Book  181288  |  reads:  686  |  chapters: 20  |  genres: Adventure, Crime, Dystopian, Fantasy
  Book  791274  |  reads:  678  |  chapters: 19  |  genres: Crime, Dystopian, Fantasy, Graphic Novel
  Book  137728  |  reads:  676  |  chapters: 18  |  genres: Crime, Dystopian, Fantasy, Graphic Novel
  Book  478798  |  reads:  675  |  chapters: 19  |  gen

In [119]:
# Evaluation with negative sampling

In [120]:
def evaluate_all_models(model_fns, test_df, train_mat,
                       n_negatives=99, Ks=[5, 10, 20], max_users=10000):
    """
    Evaluate multiple models with negative sampling.
    Computes scores ONCE per model, then evaluates at all K values.
    """
    all_book_indices = set(range(n_books))
    test_users = test_df.to_dict('records')
    sample_size = min(len(test_users), max_users)
    sampled = np.random.choice(len(test_users), sample_size, replace=False)

    # Collect rank of true item for each model and user
    model_ranks = {name: [] for name in model_fns}

    for count, i in enumerate(sampled):
        if count % 2000 == 0:
            print(f"  Processing user {count}/{sample_size}...")

        row = test_users[i]
        user = row['user_id']
        true_book = row['book_id']

        if user not in user_to_idx or true_book not in book_to_idx:
            continue

        user_idx = user_to_idx[user]
        true_book_idx = book_to_idx[true_book]
        read_indices = set(train_mat[user_idx].nonzero()[1])

        candidate_pool = list(all_book_indices - read_indices - {true_book_idx})
        if len(candidate_pool) < n_negatives:
            continue

        neg_indices = np.random.choice(candidate_pool, n_negatives, replace=False)
        eval_indices = np.concatenate([[true_book_idx], neg_indices])

        for name, fn in model_fns.items():
            try:
                recs = fn(user_idx)
            except Exception:
                model_ranks[name].append(None)
                continue

            if isinstance(recs, list):
                score_map = {book_to_idx.get(b, -1): s for b, s in recs}
            else:
                score_map = recs

            eval_scores = [score_map.get(idx, -999.0) for idx in eval_indices]
            ranked_positions = np.argsort(eval_scores)[::-1]
            rank_of_true = int(np.where(ranked_positions == 0)[0][0]) + 1
            model_ranks[name].append(rank_of_true)

    # Compute metrics at each K — one row per model with all K columns
    results = []
    for name in model_fns:
        ranks = [r for r in model_ranks[name] if r is not None]
        row = {'Model': name, 'Users': len(ranks)}
        for K in Ks:
            hits = sum(1 for r in ranks if r <= K)
            hr = hits / len(ranks) if ranks else 0
            mrr = np.mean([1.0/r if r <= K else 0.0 for r in ranks]) if ranks else 0
            ndcg = np.mean([1.0/np.log2(r+1) if r <= K else 0.0 for r in ranks]) if ranks else 0
            row[f'HR@{K}'] = f'{hr:.4f}'
            row[f'NDCG@{K}'] = f'{ndcg:.4f}'
        row['MRR'] = f"{np.mean([1.0/r for r in ranks]):.4f}" if ranks else '0.0000'
        results.append(row)

    col_order = ['Model'] + [c for K in Ks for c in [f'HR@{K}', f'NDCG@{K}']] + ['MRR', 'Users']
    return pd.DataFrame(results)[col_order]


In [121]:
# Define recommendation functions
popular_books_list = book_popularity['book_id'].tolist()


def popularity_recommend(user_idx):
    read = set(idx_to_book[b] for b in train_matrix[user_idx].nonzero()[1])
    return [(b, 1.0 / (i + 1)) for i, b in enumerate(popular_books_list) if b not in read][:200]


def cf_recommend(user_idx):
    scores = cf_scores(user_idx, train_matrix, n_recs=200)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:200]
    return [(idx_to_book[b], s) for b, s in ranked]


def cb_recommend(user_idx):
    scores = content_scores(user_idx, train_matrix, n_recs=200)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:200]
    return [(idx_to_book[b], s) for b, s in ranked]


def hybrid_recommend_fn(user_idx):
    return hybrid_recommend(user_idx, train_matrix, n_recs=200)


# --- Run evaluation (scores computed ONCE per model, all K values at once) ---
print("Evaluating with Negative Sampling (99 negatives, up to 10K users)")
print("Random baseline: HR@10 ≈ 10%")
print("=" * 80)

model_fns = {
    'Popularity': popularity_recommend,
    'CF (SVD)': cf_recommend,
    'Content-Based (TF-IDF)': cb_recommend,
    'Hybrid': hybrid_recommend_fn,
}

results_df = evaluate_all_models(model_fns, test_df, train_matrix,
                                  n_negatives=99, Ks=[5, 10, 20], max_users=10000)
print("\n")
print(results_df.to_string(index=False))


Evaluating with Negative Sampling (99 negatives, up to 10K users)
Random baseline: HR@10 ≈ 10%
  Processing user 0/10000...
  Processing user 2000/10000...
  Processing user 4000/10000...
  Processing user 6000/10000...
  Processing user 8000/10000...


                 Model   HR@5 NDCG@5  HR@10 NDCG@10  HR@20 NDCG@20    MRR  Users
            Popularity 0.1168 0.0907 0.1184  0.0913 0.1184  0.0913 0.0910  10000
              CF (SVD) 0.0943 0.0724 0.0954  0.0728 0.0954  0.0728 0.0742  10000
Content-Based (TF-IDF) 0.0589 0.0429 0.0598  0.0432 0.0598  0.0432 0.0471  10000
                Hybrid 0.0693 0.0515 0.0703  0.0519 0.0703  0.0519 0.0550  10000


In [122]:
 # Example Recommendations
example_users = [u for u in user_ids if len(user_books_list.get(u, [])) >= 5][:3]

for user in example_users:
    uidx = user_to_idx[user]
    read_books = [idx_to_book[b] for b in train_matrix[uidx].nonzero()[1]]

    print(f"\n{'═' * 65}")
    print(f"User: {user} (read {len(read_books)} books in training set)")
    print(f"{'─' * 65}")

    print("  READING HISTORY:")
    for bid in read_books[:5]:
        if bid in genre_df_binary.index:
            genres = [g for g, v in genre_df_binary.loc[bid].items() if v == 1]
            print(f"    Book {bid}  →  {', '.join(genres[:4])}")

    recs = hybrid_recommend(uidx, train_matrix, n_recs=5)
    print(f"\n  TOP 5 RECOMMENDATIONS (Hybrid):")
    for book_id, score in recs:
        if book_id in genre_df_binary.index:
            genres = [g for g, v in genre_df_binary.loc[book_id].items() if v == 1]
            print(f"    → Book {book_id}  (score: {score:.3f})  |  {', '.join(genres[:4])}")



═════════════════════════════════════════════════════════════════
User: user_0000013 (read 6 books in training set)
─────────────────────────────────────────────────────────────────
  READING HISTORY:
    Book 200976  →  Adventure, Fantasy, Graphic Novel, Historical Fiction
    Book 461083  →  Adventure, Crime, Dystopian, Fantasy
    Book 469844  →  Adventure, Graphic Novel, Humor, Literary Fiction
    Book 635456  →  Crime, Dystopian, Historical Fiction, Humor
    Book 964446  →  Adventure, Crime, Fantasy, Graphic Novel

  TOP 5 RECOMMENDATIONS (Hybrid):
    → Book 261505  (score: 0.700)  |  Adventure, Crime, Dystopian, Fantasy
    → Book 463625  (score: 0.695)  |  Adventure, Crime, Dystopian, Graphic Novel
    → Book 305611  (score: 0.463)  |  Adventure, Crime, Dystopian, Fantasy
    → Book 181288  (score: 0.462)  |  Adventure, Crime, Dystopian, Fantasy
    → Book 316800  (score: 0.462)  |  Adventure, Crime, Dystopian, Fantasy

═══════════════════════════════════════════════════════

In [123]:
# Chapter-Level Recommendation
print("Chapter-level recommendations for first example user:\n")
user = example_users[0]
recs = hybrid_recommend(user_to_idx[user], train_matrix, n_recs=5)
for book_id, score in recs:
    ch_id, ch_seq = recommend_chapter_fast(user, book_id)
    print(f"  Book {book_id} → Start at chapter {ch_seq} (chapter_id: {ch_id})")

print(f"\n{'─' * 50}")
print("\nCold-start recommendations (new user):\n")

for pref in [None, 'Fantasy', 'Horror']:
    label = pref if pref else 'No preference (popularity)'
    print(f"  Genre: {label}")
    for bid in cold_start_recommend(genre_preference=pref, n_recs=3):
        pop = book_popularity[book_popularity['book_id'] == bid]['read_count'].values[0]
        print(f"    → Book {bid}  (reads: {pop})")
    print()


Chapter-level recommendations for first example user:

  Book 261505 → Start at chapter 1 (chapter_id: 6476736)
  Book 463625 → Start at chapter 1 (chapter_id: 1471043)
  Book 305611 → Start at chapter 1 (chapter_id: 6879309)
  Book 181288 → Start at chapter 1 (chapter_id: 3079814)
  Book 316800 → Start at chapter 1 (chapter_id: 9548426)

──────────────────────────────────────────────────

Cold-start recommendations (new user):

  Genre: No preference (popularity)
    → Book 672233  (reads: 999)
    → Book 760434  (reads: 818)
    → Book 615276  (reads: 807)

  Genre: Fantasy
    → Book 672233  (reads: 999)
    → Book 760434  (reads: 818)
    → Book 615276  (reads: 807)

  Genre: Horror
    → Book 672233  (reads: 999)
    → Book 760434  (reads: 818)
    → Book 615276  (reads: 807)

